# 🤖 Jarvis 3D — Setup

Run this notebook **first** before any other notebook.

### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Add your OpenAI API key to **Colab Secrets** (key icon 🔑 on left sidebar) with name `OPENAI_API_KEY`
3. Run all cells top to bottom

In [9]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — go to Runtime > Change runtime type > T4 GPU')

Sat May  9 03:33:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# Install core dependencies (pinned for Colab Python 3.12 compatibility)
!pip install -q --upgrade pip setuptools wheel
!pip install -q --force-reinstall "numpy==1.26.4"
!pip install -q "openai>=1.30.0" "openai-whisper==20240930" "gTTS>=2.5.1" pydub SpeechRecognition
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print('Core packages installed with compatible versions.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.17.3 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.17.3 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.17.3 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
accelerate 1.13.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.17.3 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you ha

In [11]:
# Install TripoSR (single image -> 3D model)
import os
if not os.path.exists('/content/TripoSR'):
    !git clone https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
%cd /content/TripoSR
!pip install -q -r requirements.txt
!pip install -q --force-reinstall "numpy==1.26.4"
print('TripoSR ready and NumPy re-pinned for compatibility.')

/content/TripoSR
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.3 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 11.0.3 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.17.3 which is incompatible.
diffusers 0.37.1 requires hug

In [12]:
# Install audio utilities (needed for Colab mic recording)
!apt-get install -y -q ffmpeg portaudio19-dev
!pip install -q pyaudio
print('Audio utilities installed.')

Reading package lists...
Building dependency tree...
Reading state information...
portaudio19-dev is already the newest version (19.6.0-1.1).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
Audio utilities installed.


In [13]:
# Verify OpenAI API key from Colab Secrets
from google.colab import userdata
try:
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key and len(api_key) > 10:
        print('OpenAI API key found.')
    else:
        print('WARNING: API key seems empty. Add OPENAI_API_KEY in Colab Secrets (left sidebar key icon).')
except Exception as e:
    print(f'Could not read secret: {e}')
    print('Manually set: import os; os.environ["OPENAI_API_KEY"] = "sk-..."')

OpenAI API key found.


In [14]:
# If this is the first run after installs, restart once so compiled extensions reload cleanly
import os
print('If you saw numpy binary mismatch errors, run this cell once to restart runtime.')
print('After restart, run all cells from top again.')
# Uncomment the next line only when needed:
# os.kill(os.getpid(), 9)

If you saw numpy binary mismatch errors, run this cell once to restart runtime.
After restart, run all cells from top again.


In [1]:
# Final import test
!pip install gtts
!pip install whisper
import torch
import openai
from gtts import gTTS

try:
    import whisper
except Exception as e:
    print('Whisper import failed. This is usually a runtime-state issue after package installs.')
    print(f'Error: {e}')
    print('Run the restart cell, then run all cells from top again.')
    raise

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print('All imports successful. Ready to run other notebooks!')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
All imports successful. Ready to run other notebooks!
